In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

# ============================================================
# Config
# ============================================================
BASE_DIR = Path("./")
INPUT_GFEVD = BASE_DIR / "gfevd_all.npy"
INPUT_DATE = BASE_DIR / "tvpvar_input_scaled.csv"
INPUT_EVENTS = BASE_DIR / "events_window0.csv"

OUT_DIR = BASE_DIR / "event_quantification_outputs"
OUT_DIR.mkdir(exist_ok=True)

# 변수명 직접 지정
VAR_NAMES = [
    "SOLVPN",
    "COPPER",
    "GOLD",
    "SILVER",
    "DXY",
    "UST10Y",
    "VIX"
]

# ============================================================
# 1. Load
# ============================================================
gfevd = np.load(INPUT_GFEVD)   # shape: (T, k, k)

df_date = pd.read_csv(INPUT_DATE)
df_date["Date"] = pd.to_datetime(df_date["Date"])

df_events = pd.read_csv(INPUT_EVENTS)
df_events["Date"] = pd.to_datetime(df_events["Date"])

T, k, _ = gfevd.shape

if len(VAR_NAMES) != k:
    raise ValueError(f"VAR_NAMES length ({len(VAR_NAMES)}) != gfevd dimension ({k})")

# gfevd 길이에 맞게 날짜 정렬
dates = pd.to_datetime(df_date["Date"].values[-T:])
date_to_idx = {pd.Timestamp(d): i for i, d in enumerate(dates)}

print(f"[INFO] T={T}, k={k}")
print(f"[INFO] Loaded {len(df_events)} events")

# ============================================================
# 2. Delta 계산
# ============================================================
# delta[t] = |gfevd[t] - gfevd[t-1]|, shape: (T-1, k, k)
delta = np.abs(gfevd[1:] - gfevd[:-1])

# 자기 자신 제외 mask
offdiag_mask = ~np.eye(k, dtype=bool)

# 날짜도 delta에 맞게 하나 뒤부터 사용
delta_dates = dates[1:]

# ============================================================
# 3. Event Quantification
# ============================================================
summary_rows = []
pairwise_rows = []

for event_date in df_events["Date"]:
    event_date = pd.Timestamp(event_date)

    if event_date not in date_to_idx:
        print(f"[WARN] Event date {event_date.date()} not found in aligned dates, skipped")
        continue

    idx = date_to_idx[event_date]

    # delta는 t=1부터 존재하므로 idx=0이면 계산 불가
    if idx == 0:
        print(f"[WARN] Event date {event_date.date()} has no previous point for delta, skipped")
        continue

    delta_idx = idx - 1
    delta_mat = delta[delta_idx]  # (k, k)

    # 전체 magnitude
    magnitude = delta_mat[offdiag_mask].sum()

    # 상위 pair 찾기
    tmp_pairs = []
    for i in range(k):
        for j in range(k):
            if i == j:
                continue
            tmp_pairs.append((i, j, delta_mat[i, j]))

    tmp_pairs_sorted = sorted(tmp_pairs, key=lambda x: x[2], reverse=True)

    top1_i, top1_j, top1_val = tmp_pairs_sorted[0]
    top2_i, top2_j, top2_val = tmp_pairs_sorted[1]
    top3_i, top3_j, top3_val = tmp_pairs_sorted[2]

    summary_rows.append({
        "Date": event_date,
        "Magnitude": magnitude,
        "Top1_From": VAR_NAMES[top1_i],
        "Top1_To": VAR_NAMES[top1_j],
        "Top1_DeltaS": top1_val,
        "Top2_From": VAR_NAMES[top2_i],
        "Top2_To": VAR_NAMES[top2_j],
        "Top2_DeltaS": top2_val,
        "Top3_From": VAR_NAMES[top3_i],
        "Top3_To": VAR_NAMES[top3_j],
        "Top3_DeltaS": top3_val,
    })

    for i in range(k):
        for j in range(k):
            if i == j:
                continue
            pairwise_rows.append({
                "Date": event_date,
                "From": VAR_NAMES[i],
                "To": VAR_NAMES[j],
                "DeltaS": delta_mat[i, j]
            })

# ============================================================
# 4. Save
# ============================================================
df_summary = pd.DataFrame(summary_rows).sort_values("Date").reset_index(drop=True)
df_pairwise = pd.DataFrame(pairwise_rows).sort_values(["Date", "DeltaS"], ascending=[True, False]).reset_index(drop=True)

summary_path = OUT_DIR / "event_quantification_window0_summary.csv"
pairwise_path = OUT_DIR / "event_quantification_window0_pairwise_long.csv"

df_summary.to_csv(summary_path, index=False)
df_pairwise.to_csv(pairwise_path, index=False)

print(f"[INFO] Saved summary: {summary_path}")
print(f"[INFO] Saved pairwise: {pairwise_path}")
print("[INFO] Done")